In [1]:
import os
import getpass
import sqlite3
import uuid
import random
import re
from typing import TypedDict, Optional

from pydantic import BaseModel, Field

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END

In [2]:
def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ").strip()

_set_env("GOOGLE_API_KEY")
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0)

GOOGLE_API_KEY:  ········


In [3]:
def extract_text(agent_resp) -> str:
    """Return the final agent message as a clean string.

    Gemini (and the LangChain 1.0 message format) can return message .content
    as a LIST of content blocks, e.g. [{"type": "text", "text": "..."}],
    instead of a plain string. Calling .strip() on a list raises
    AttributeError, so this normalizes both shapes to a string.
    """
    if isinstance(agent_resp, dict) and "messages" in agent_resp:
        content = agent_resp["messages"][-1].content
    else:
        return str(agent_resp).strip()
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, dict):
                parts.append(block.get("text", ""))
            elif isinstance(block, str):
                parts.append(block)
        return "".join(parts).strip()

    return str(content).strip()
_UUID_RE = re.compile(
    r"[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}"
)


In [4]:
def extract_order_id(agent_resp) -> str:
    """Pull the bare order_id (UUID) out of the agent's reply.

    create_agent with Gemini tends to wrap the tool result in a full sentence
    ("...Your order ID is <uuid>."), so we extract just the UUID. Falls back to
    the full text only if no UUID is present.
    """
    text = extract_text(agent_resp)
    match = _UUID_RE.search(text)
    return match.group(0) if match else text

In [5]:
def payment_failed(agent_resp) -> bool:
    """Reliably detect a payment failure.

    The payment tool returns a raw "FAIL:..." or "SUCCESS" string, but the LLM
    then paraphrases it ("The payment failed because..."), so checking only the
    final message text is unreliable. We scan ALL messages -- including the raw
    ToolMessage output -- for the failure signal.
    """
    if isinstance(agent_resp, dict) and "messages" in agent_resp:
        for msg in agent_resp["messages"]:
            content = getattr(msg, "content", "")
            if isinstance(content, list):
                content = " ".join(
                    b.get("text", "") if isinstance(b, dict) else str(b)
                    for b in content
                )
            if "FAIL" in str(content).upper():
                return True
            return False
    return "FAIL" in extract_text(agent_resp).upper()

In [6]:
def delete_order(agent_rest) -> str:
    """Check for the message from payment failed function if the tool says FAIL or fail or anything apart from
    SUCCESS delete, delete the payment entirely """
    status = str(agent_rest.get("status", "")).strip().upper()

    if status != "SUCCESS":
        order_id = agent_rest.get("order_id")
        if not order_id:
            return "FAILED: No order_id found, cannot delete payment."

        try:
            delete_response = payment_gateway.delete_payment(order_id)  # placeholder
            if delete_response.get("success"):
                return f"Payment for order {order_id} deleted due to status: {status}"
            else:
                return f"FAILED: Could not delete payment for order {order_id}"
        except Exception as e:
            return f"ERROR: Exception while deleting payment - {str(e)}"

    return "Payment SUCCESS, no deletion needed."

In [7]:
conn = sqlite3.connect("orders.db", check_same_thread=False)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS orders (
    order_id TEXT PRIMARY KEY,
    item TEXT,
    amount REAL,
    payment_status TEXT,
    notification_status TEXT
)
""")	
conn.commit()

In [8]:
class CreateOrderInput(BaseModel):
    item: str = Field(description="Item being ordered")
    amount: float = Field(description="Order amount")

class PaymentInput(BaseModel):
    order_id: str = Field(description="Order ID for payment")

In [9]:
class NotifyInput(BaseModel):
    order_id: str = Field(description="Order ID to notify")

class DeletePayment(BaseModel):
    order_id: str = Field(description="Order ID to delete")
    
def create_order_tool(data: CreateOrderInput):
    order_id = str(uuid.uuid4())
    cursor.execute(
        "INSERT INTO orders VALUES (?, ?, ?, ?, ?)",
        (order_id, data.item, data.amount, "PENDING", "NOT_SENT")
    )
    conn.commit()
    return {"order_id": order_id}

def process_payment_tool(data: PaymentInput):
    cursor.execute(
        "UPDATE orders SET payment_status = ? WHERE order_id = ?",
        ("SUCCESS", data.order_id)
    )
    conn.commit()
    return {"payment_status": "SUCCESS"}

def notify_user_tool(data: NotifyInput):
    cursor.execute(
        "UPDATE orders SET notification_status = ? WHERE order_id = ?",
        ("SENT", data.order_id)
    )

def delete_payment_tool(data: DeletePayment):
    cursor.execute(
        "DELETE FROM orders WHERE order_id = ?",
        (data.order_id,)
    )
    conn.commit()
    return {"status": "DELETED", "order_id": data.order_id}

In [10]:
@tool
def create_order_tool_agent(item: str, amount: float = 100.0) -> str:
    """Create an order and return the generated order_id"""
    result = create_order_tool(CreateOrderInput(item=item, amount=amount))
    return result["order_id"]

@tool
def process_payment_tool_agent(order_id: str) -> str:
    """Process payment for a given order_id"""
    result = process_payment_tool(PaymentInput(order_id=order_id))
    return result["payment_status"]

@tool
def notify_user_tool_agent(order_id: str) -> str:
    """Notify user for a given order_id"""
    result = notify_user_tool(NotifyInput(order_id=order_id))
    return result["notification"]

@tool
def delete_payment_tool_agent(order_id: str)->str:
    """Order deleted"""
    result = delete_payment_tool(DeletePayment(order_id=order_id))
    return result["status"]

In [11]:
order_agent = create_agent(
    model=llm,
    tools=[create_order_tool_agent],
    system_prompt="""
You are an Order Service Agent. When given an item name and optional amount,
return the order_id string.
"""
)

payment_agent = create_agent(
    model=llm,
    tools=[process_payment_tool_agent],
    system_prompt="""
You are a Payment Service Agent. When given an order_id, return PAYMENT_SUCCESS or FAIL.
"""
)

notification_agent = create_agent(
    model=llm,
    tools=[notify_user_tool_agent],
    system_prompt="""
You are a Notification Service Agent. When given an order_id, return NOTIFICATION_SENT.
"""
)

delete_agent = create_agent(
    model = llm,
    tools=[delete_payment_tool_agent],
    system_prompt="""
You are a Delete service Agent. When payment is failed, return the delete message.
"""
)


In [12]:

class WorkflowState(TypedDict):
    user_input: str
    order_id: Optional[str]
    payment_status: Optional[str]
    notification_status: Optional[str]
    error: Optional[str]
    delete_status: Optional[str]
    retries: int

MAX_RETRIES = 2

def order_requester(state: WorkflowState):
    print("[OrderRequester] Running")

    try:
        item = state["user_input"]
        amount = 100.0
        agent_resp = order_agent.invoke({
            "messages": [{"role": "user", "content": f"Create order for {item} with amount {amount}."}]
        })
        order_id = extract_order_id(agent_resp)
        return {
            "order_id": order_id,
            "retries": state["retries"]
        }

    except Exception as e:
        return {"error": str(e), "retries": state["retries"] + 1}

def payment_processor(state: WorkflowState):
    print("[PaymentProcessor] Running")

    try:
        agent_resp = payment_agent.invoke({
            "messages": [{"role": "user", "content": f"Process payment for order {state['order_id']}"}]
        })

        payment_status = extract_text(agent_resp)

        return {
            "payment_status": payment_status,
            "retries": state["retries"]
        }

    except Exception as e:
        return {"error": str(e), "retries": state["retries"] + 1}

def notifier(state: WorkflowState):
    print("[Notifier] Running")

    try:
        agent_resp = notification_agent.invoke({
            "messages": [{"role": "user", "content": f"Notify user for order {state['order_id']}"}]
        })

        notification_status = extract_text(agent_resp)

        return {
            "notification_status": notification_status,
            "retries": state["retries"]
        }

    except Exception as e:
        return {"error": str(e), "retries": state["retries"] + 1}
        
def check_error(state: WorkflowState):
    if state.get("error"):
        if state["retries"] >= MAX_RETRIES:
            print("[X] Max retries reached. Ending workflow.")
            return END
        print("[Retry] Retrying...")
        return "order_requester"
    return "payment_processor"

def delete_node(state: WorkflowState):
    print("[DeleteNode] Payment failed — deleting order")
    try:
        agent_resp = delete_agent.invoke({
            "messages": [{"role": "user", "content": f"Delete order {state['order_id']} because payment failed."}]
        })
        delete_status = extract_text(agent_resp)
        return {"delete_status": delete_status}
    except Exception as e:
        return {"error": str(e), "retries": state["retries"] + 1}
        
def check_payment(state: WorkflowState):
    if "FAIL" in str(state.get("payment_status", "")).upper():
        return "delete_node"
    return "notifier"

def process_payment_tool(data: PaymentInput):
    status = "FAIL" if random.random() < 0.5 else "SUCCESS"
    cursor.execute(
        "UPDATE orders SET payment_status = ? WHERE order_id = ?",
        (status, data.order_id)
    )
    conn.commit()
    return {"payment_status": status}


In [13]:
builder = StateGraph(WorkflowState)

builder.add_node("order_requester", order_requester)
builder.add_node("payment_processor", payment_processor)
builder.add_node("notifier", notifier)

builder.set_entry_point("order_requester")

builder.add_conditional_edges("order_requester", check_error)
builder.add_node("delete_node", delete_node)
builder.add_conditional_edges("payment_processor", check_payment)
builder.add_edge("delete_node", END)

builder.add_edge("notifier", END)

app = builder.compile()


In [18]:
# Force payment to always FAIL for testing
def process_payment_tool(data: PaymentInput):
    cursor.execute(
        "UPDATE orders SET payment_status = ? WHERE order_id = ?",
        ("FAIL", data.order_id)
    )
    conn.commit()
    return {"payment_status": "FAIL"}


In [21]:
def human_approval(state: WorkflowState):
    print("\n[Human] Review Required")
    print("Reason:", state["error"])
    print("Retry Payment? (yes/no)")

    decision = input(">> ").strip().lower()

    return {
        "human_decision": decision
    }

In [22]:

result = app.invoke({
    "user_input": "Laptop Purchase",
    "order_id": None,
    "payment_status": None,
    "notification_status": None,
    "delete_status" : None,
    "error": None,
    "retries": 0
})

[OrderRequester] Running
[PaymentProcessor] Running
[DeleteNode] Payment failed — deleting order


In [23]:
print(result)

{'user_input': 'Laptop Purchase', 'order_id': '510f97eb-90ec-435d-9c55-e4b76a0e4a3d', 'payment_status': 'FAIL', 'notification_status': None, 'error': None, 'delete_status': 'The order 510f97eb-90ec-435d-9c55-e4b76a0e4a3d has been successfully deleted due to the payment failure.', 'retries': 0}
